In [1]:
# ============================================================
# 08_prediction.ipynb
# Cell 1: Imports
# ============================================================

import os
import gc
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

print("=" * 70)
print("GLOBAL PRICE WATCH — PREDICTION")
print("=" * 70)

print("TensorFlow:", tf.__version__)
print("✓ Imports completed")

GLOBAL PRICE WATCH — PREDICTION
TensorFlow: 2.21.0
✓ Imports completed


In [3]:
# ============================================================
# Cell 2: File paths
# ============================================================

MODEL_FILE = "leakage_safe_gru_attention_model.keras"

SEQUENCE_SCALER_FILE = "leakage_safe_sequence_scaler.pkl"
STATIC_SCALER_FILE = "leakage_safe_static_scaler.pkl"
TARGET_SCALER_FILE = "leakage_safe_target_scaler.pkl"

DATA_FILE = "inflation_feature_engineered_leakage_safe.csv"

required_files = [
    MODEL_FILE,
    SEQUENCE_SCALER_FILE,
    STATIC_SCALER_FILE,
    TARGET_SCALER_FILE,
    DATA_FILE
]

print("=" * 70)
print("CHECKING REQUIRED FILES")
print("=" * 70)

for file in required_files:
    if os.path.exists(file):
        print("✓", file)
    else:
        print("✗ MISSING:", file)

missing_files = [
    file for file in required_files
    if not os.path.exists(file)
]

if missing_files:
    raise FileNotFoundError(
        f"\nMissing files: {missing_files}"
    )

print("\n✓ ALL REQUIRED FILES AVAILABLE")

CHECKING REQUIRED FILES
✓ leakage_safe_gru_attention_model.keras
✓ leakage_safe_sequence_scaler.pkl
✓ leakage_safe_static_scaler.pkl
✓ leakage_safe_target_scaler.pkl
✓ inflation_feature_engineered_leakage_safe.csv

✓ ALL REQUIRED FILES AVAILABLE


In [4]:
# ============================================================
# Cell 3: Load trained model and scalers
# ============================================================

print("=" * 70)
print("LOADING TRAINED MODEL AND SCALERS")
print("=" * 70)

prediction_model = tf.keras.models.load_model(
    MODEL_FILE,
    compile=False
)

sequence_scaler = joblib.load(
    SEQUENCE_SCALER_FILE
)

static_scaler = joblib.load(
    STATIC_SCALER_FILE
)

target_scaler = joblib.load(
    TARGET_SCALER_FILE
)

print("✓ Model loaded")
print("✓ Sequence scaler loaded")
print("✓ Static scaler loaded")
print("✓ Target scaler loaded")

print("\nModel:", prediction_model.name)

print("\nExpected inputs:")

for inp in prediction_model.inputs:
    print(inp.name, "->", inp.shape)

LOADING TRAINED MODEL AND SCALERS
✓ Model loaded
✓ Sequence scaler loaded
✓ Static scaler loaded
✓ Target scaler loaded

Model: Leakage_Safe_GRU_MultiHead_Temporal_Attention

Expected inputs:
sequence_input -> (None, 10, 1)
static_input -> (None, 77)


In [5]:
# ============================================================
# Cell 4: Load leakage-safe feature dataset
# ============================================================

modeling_df = pd.read_csv(DATA_FILE)

print("=" * 70)
print("LOADING LEAKAGE-SAFE DATASET")
print("=" * 70)

print("Shape:", modeling_df.shape)

print("\nCountries:", modeling_df["Country Name"].nunique())

print("\nYear columns:")

year_columns = [
    col for col in modeling_df.columns
    if str(col).isdigit()
]

print(
    f"{year_columns[0]} - {year_columns[-1]}"
)

print("\n✓ DATASET LOADED")

LOADING LEAKAGE-SAFE DATASET
Shape: (193, 145)

Countries: 193

Year columns:
1960 - 2025

✓ DATASET LOADED


In [6]:
# ============================================================
# Cell 5: Define static features
# ============================================================

id_columns = [
    "Country Name",
    "Country Code"
]

year_columns = [
    col for col in modeling_df.columns
    if str(col).isdigit()
]

excluded_columns = (
    id_columns
    + year_columns
)

static_features = [
    col for col in modeling_df.columns
    if col not in excluded_columns
]

print("=" * 70)
print("STATIC FEATURES")
print("=" * 70)

print("Static feature count:", len(static_features))

for i, feature in enumerate(static_features, 1):
    print(f"{i:2d}. {feature}")

if len(static_features) != 77:
    raise ValueError(
        f"Expected 77 static features, "
        f"found {len(static_features)}"
    )

print("\n✓ EXACTLY 77 STATIC FEATURES FOUND")

STATIC FEATURES
Static feature count: 77
 1. Mean_Inflation
 2. Median_Inflation
 3. Std_Inflation
 4. Min_Inflation
 5. Max_Inflation
 6. Inflation_Range
 7. Mean_Recent_5Y
 8. Mean_Recent_10Y
 9. Mean_Previous_10Y
10. Long_Term_Change
11. Long_Term_Trend
12. Inflation_Volatility
13. High_Inflation_Years
14. Very_High_Inflation_Years
15. Negative_Inflation_Years
16. Diff_1961
17. Diff_1962
18. Diff_1963
19. Diff_1964
20. Diff_1965
21. Diff_1966
22. Diff_1967
23. Diff_1968
24. Diff_1969
25. Diff_1970
26. Diff_1971
27. Diff_1972
28. Diff_1973
29. Diff_1974
30. Diff_1975
31. Diff_1976
32. Diff_1977
33. Diff_1978
34. Diff_1979
35. Diff_1980
36. Diff_1981
37. Diff_1982
38. Diff_1983
39. Diff_1984
40. Diff_1985
41. Diff_1986
42. Diff_1987
43. Diff_1988
44. Diff_1989
45. Diff_1990
46. Diff_1991
47. Diff_1992
48. Diff_1993
49. Diff_1994
50. Diff_1995
51. Diff_1996
52. Diff_1997
53. Diff_1998
54. Diff_1999
55. Diff_2000
56. Diff_2001
57. Diff_2002
58. Diff_2003
59. Diff_2004
60. Diff_2005
61. 

In [7]:
# ============================================================
# Cell 6: Verify static feature order
# ============================================================

print("=" * 70)
print("VERIFYING STATIC FEATURE ORDER")
print("=" * 70)

expected_static_count = static_scaler.n_features_in_

print(
    "Scaler expects:",
    expected_static_count
)

print(
    "Dataset provides:",
    len(static_features)
)

if expected_static_count != len(static_features):
    raise ValueError(
        "Static feature count does not match saved scaler!"
    )

print("\n✓ STATIC FEATURE COUNT MATCHES SCALER")

VERIFYING STATIC FEATURE ORDER
Scaler expects: 77
Dataset provides: 77

✓ STATIC FEATURE COUNT MATCHES SCALER


In [7]:
# ============================================================
# Cell 7: Identify inflation year columns
# ============================================================

year_columns = [
    col for col in modeling_df.columns
    if str(col).isdigit()
    and 1960 <= int(col) <= 2025
]

year_columns = sorted(
    year_columns,
    key=int
)

print("=" * 60)
print("YEAR COLUMNS")
print("=" * 60)

print("Number of years:", len(year_columns))
print("First year:", year_columns[0])
print("Last year:", year_columns[-1])

YEAR COLUMNS
Number of years: 66
First year: 1960
Last year: 2025


In [8]:
# ============================================================
# Cell 7: Prediction function
# ============================================================

def predict_country_inflation(
    country_name,
    target_year,
    dataframe=modeling_df
):
    """
    Predict inflation for a country and target year.

    Uses:
    - Previous 10 years of inflation
    - 77 leakage-safe static features
    - Trained GRU + Multi-Head Temporal Attention model
    """

    # --------------------------------------------------------
    # Check country
    # --------------------------------------------------------

    country_data = dataframe[
        dataframe["Country Name"] == country_name
    ]

    if country_data.empty:
        raise ValueError(
            f"Country '{country_name}' not found."
        )

    country_data = country_data.iloc[0]

    # --------------------------------------------------------
    # Check target year
    # --------------------------------------------------------

    target_year = int(target_year)

    if str(target_year) not in year_columns:
        raise ValueError(
            f"Year {target_year} is not available."
        )

    # --------------------------------------------------------
    # Need previous 10 years
    # --------------------------------------------------------

    required_history = list(
        range(target_year - 10, target_year)
    )

    missing_years = [
        year
        for year in required_history
        if str(year) not in year_columns
    ]

    if missing_years:
        raise ValueError(
            f"Insufficient history. Missing years: "
            f"{missing_years}"
        )

    # --------------------------------------------------------
    # Create 10-year sequence
    # --------------------------------------------------------

    sequence_values = np.array([
        country_data[str(year)]
        for year in required_history
    ], dtype=np.float32)

    sequence_values = sequence_values.reshape(
        1, 10, 1
    )

    # --------------------------------------------------------
    # Create static feature vector
    # --------------------------------------------------------

    static_values = np.array([
        country_data[feature]
        for feature in static_features
    ], dtype=np.float32)

    static_values = static_values.reshape(
        1, -1
    )

    # --------------------------------------------------------
    # Scale inputs
    # --------------------------------------------------------

    sequence_scaled = sequence_scaler.transform(
        sequence_values.reshape(-1, 1)
    ).reshape(1, 10, 1)

    static_scaled = static_scaler.transform(
        static_values
    )

    # --------------------------------------------------------
    # Model prediction
    # --------------------------------------------------------

    prediction_scaled = prediction_model.predict(
        [
            sequence_scaled,
            static_scaled
        ],
        verbose=0
    )

    # --------------------------------------------------------
    # Convert back to inflation %
    # --------------------------------------------------------

    prediction = target_scaler.inverse_transform(
        prediction_scaled
    ).ravel()[0]

    # --------------------------------------------------------
    # Result
    # --------------------------------------------------------

    result = {
        "Country": country_name,
        "Country_Code": country_data["Country Code"],
        "Target_Year": target_year,
        "History_Used": (
            f"{required_history[0]}-{required_history[-1]}"
        ),
        "Predicted_Inflation": float(prediction)
    }

    return result

In [9]:
# ============================================================
# Cell 8: Test prediction
# ============================================================

TEST_COUNTRY = "India"
TEST_YEAR = 2025

result = predict_country_inflation(
    TEST_COUNTRY,
    TEST_YEAR
)

print("=" * 70)
print("SINGLE COUNTRY PREDICTION")
print("=" * 70)

for key, value in result.items():
    print(f"{key}: {value}")

SINGLE COUNTRY PREDICTION
Country: India
Country_Code: IND
Target_Year: 2025
History_Used: 2015-2024
Predicted_Inflation: 7.366077899932861


In [10]:
# ============================================================
# Cell 9: Multiple country predictions
# ============================================================

countries_to_predict = [
    "India",
    "United States",
    "China",
    "Germany",
    "Brazil"
]

target_year = 2025

prediction_results = []

for country in countries_to_predict:

    try:
        result = predict_country_inflation(
            country_name=country,
            target_year=target_year
        )

        prediction_results.append(result)

    except Exception as e:

        print(f"⚠️ Could not predict {country}: {e}")


predictions_df = pd.DataFrame(
    prediction_results
)

print("=" * 70)
print("MULTI-COUNTRY PREDICTIONS")
print("=" * 70)

print(predictions_df.to_string(index=False))

MULTI-COUNTRY PREDICTIONS
      Country Country_Code  Target_Year History_Used  Predicted_Inflation
        India          IND         2025    2015-2024             7.366078
United States          USA         2025    2015-2024             3.641899
        China          CHN         2025    2015-2024             2.859625
      Germany          DEU         2025    2015-2024             3.357089
       Brazil          BRA         2025    2015-2024             7.491090


In [12]:
# ============================================================
# Cell 10: Predict all countries
# ============================================================

TARGET_YEAR = 2025

all_predictions = []

print("=" * 70)
print(f"PREDICTING INFLATION FOR ALL COUNTRIES — {TARGET_YEAR}")
print("=" * 70)

for _, row in modeling_df.iterrows():

    country = row["Country Name"]

    try:

        result = predict_country_inflation(
            country_name=country,
            target_year=TARGET_YEAR
        )

        all_predictions.append(result)

    except Exception as e:

        print(
            f"⚠️ Failed: {country} -> {e}"
        )


all_predictions_df = pd.DataFrame(
    all_predictions
)

print("\n" + "=" * 70)
print("ALL-COUNTRY PREDICTION COMPLETED")
print("=" * 70)

print(
    "Countries predicted:",
    len(all_predictions_df)
)

print(
    "Expected countries:",
    modeling_df["Country Name"].nunique()
)

print("\nPreview:")
print(
    all_predictions_df.head(10).to_string(
        index=False
    )
)

PREDICTING INFLATION FOR ALL COUNTRIES — 2025

ALL-COUNTRY PREDICTION COMPLETED
Countries predicted: 193
Expected countries: 193

Preview:
             Country Country_Code  Target_Year History_Used  Predicted_Inflation
               Aruba          ABW         2025    2015-2024             3.885369
         Afghanistan          AFG         2025    2015-2024             6.418882
              Angola          AGO         2025    2015-2024            23.409519
             Albania          ALB         2025    2015-2024             3.999496
United Arab Emirates          ARE         2025    2015-2024             6.268651
           Argentina          ARG         2025    2015-2024           158.149414
             Armenia          ARM         2025    2015-2024             5.827451
 Antigua and Barbuda          ATG         2025    2015-2024             4.047171
           Australia          AUS         2025    2015-2024             4.336691
             Austria          AUT         2025    2

In [13]:
# ============================================================
# Cell 11: Save all-country predictions
# ============================================================

OUTPUT_FILE = "global_inflation_predictions_2025.csv"

all_predictions_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("=" * 70)
print("ALL-COUNTRY PREDICTIONS SAVED")
print("=" * 70)

print("File:", OUTPUT_FILE)
print("Shape:", all_predictions_df.shape)

print("\nColumns:")
print(all_predictions_df.columns.tolist())

print("\n✓ PREDICTION FILE SAVED SUCCESSFULLY")

ALL-COUNTRY PREDICTIONS SAVED
File: global_inflation_predictions_2025.csv
Shape: (193, 5)

Columns:
['Country', 'Country_Code', 'Target_Year', 'History_Used', 'Predicted_Inflation']

✓ PREDICTION FILE SAVED SUCCESSFULLY


In [14]:
# ============================================================
# Cell 12: Rank countries by predicted inflation
# ============================================================

ranked_predictions = all_predictions_df.sort_values(
    by="Predicted_Inflation",
    ascending=False
).reset_index(drop=True)

ranked_predictions.insert(
    0,
    "Rank",
    np.arange(1, len(ranked_predictions) + 1)
)

print("=" * 70)
print("TOP 20 COUNTRIES BY PREDICTED INFLATION — 2025")
print("=" * 70)

print(
    ranked_predictions[
        [
            "Rank",
            "Country",
            "Country_Code",
            "Predicted_Inflation"
        ]
    ].head(20).to_string(index=False)
)

TOP 20 COUNTRIES BY PREDICTED INFLATION — 2025
 Rank               Country Country_Code  Predicted_Inflation
    1         Venezuela, RB          VEN           265.613556
    2             Argentina          ARG           158.149414
    3                 Sudan          SDN            97.925224
    4              Zimbabwe          ZWE            61.338543
    5               Lebanon          LBN            60.542042
    6               Turkiye          TUR            49.909649
    7              Suriname          SUR            27.103878
    8    West Bank and Gaza          PSE            26.919521
    9    Iran, Islamic Rep.          IRN            26.583172
   10          Sierra Leone          SLE            26.427948
   11                Malawi          MWI            25.418381
   12                Angola          AGO            23.409519
   13               Nigeria          NGA            23.369907
   14                 Haiti          HTI            23.153950
   15                 G

In [16]:
# ============================================================
# Cell 13: Save ranked predictions
# ============================================================

RANKED_OUTPUT_FILE = (
    "global_inflation_predictions_2025_ranked.csv"
)

ranked_predictions.to_csv(
    RANKED_OUTPUT_FILE,
    index=False
)

print("=" * 70)
print("RANKED PREDICTIONS SAVED")
print("=" * 70)

print("File:", RANKED_OUTPUT_FILE)
print("Rows:", len(ranked_predictions))

print("\n✓ RANKED PREDICTION FILE SAVED")

RANKED PREDICTIONS SAVED
File: global_inflation_predictions_2025_ranked.csv
Rows: 193

✓ RANKED PREDICTION FILE SAVED


In [17]:
# ============================================================
# Cell 14: Prediction sanity check
# ============================================================

print("=" * 70)
print("PREDICTION SANITY CHECK")
print("=" * 70)

print("Total predictions:", len(all_predictions_df))

print(
    "Missing predictions:",
    all_predictions_df["Predicted_Inflation"].isna().sum()
)

print(
    "Minimum prediction:",
    f"{all_predictions_df['Predicted_Inflation'].min():.4f}%"
)

print(
    "Maximum prediction:",
    f"{all_predictions_df['Predicted_Inflation'].max():.4f}%"
)

print(
    "Mean prediction:",
    f"{all_predictions_df['Predicted_Inflation'].mean():.4f}%"
)

print(
    "Median prediction:",
    f"{all_predictions_df['Predicted_Inflation'].median():.4f}%"
)

if all_predictions_df["Predicted_Inflation"].isna().sum() != 0:
    raise ValueError("Missing predictions detected!")

print("\n✓ ALL 193 PREDICTIONS ARE VALID")

PREDICTION SANITY CHECK
Total predictions: 193
Missing predictions: 0
Minimum prediction: 0.6208%
Maximum prediction: 265.6136%
Mean prediction: 10.2272%
Median prediction: 5.2466%

✓ ALL 193 PREDICTIONS ARE VALID
